# 01 — Erros de impressão: frequências e medidas descritivas

Percurso: enunciado → dados e rol → frequências individuais → classes → medidas → interpretação.


1- Contou-se o número de erros de impressão da primeira página de um jornal durante 50 dias, obtendo-se os resultados abaixo:

| 1 | 2  | 3 | 4  | 5  | 6  | 7  | 8  | 9  | 10 |
|--:|---:|--:|---:|---:|---:|---:|---:|---:|---:|
| 8 | 11 | 8 | 12 | 14 | 13 | 11 | 14 | 14 | 15 |
| 6 | 10 | 14 | 19 | 6 | 12 | 7 | 5 | 8 | 8 |
| 10 | 16 | 10 | 12 | 12 | 8 | 11 | 6 | 7 | 12 |
| 7 | 10 | 14 | 5 | 12 | 7 | 9 | 12 | 11 | 9 |
| 14 | 8 | 14 | 8 | 12 | 10 | 12 | 22 | 7 | 15 |


 - Calcule as frequências relativas e as frequências relativas acumuladas. Considere a representação individual dos erros e em intervalos de 5 valores
 - Calcule a média, desvio-padrão e assimetria;  
 - Qual a frequência de valores maiores que 10? Qual a frequência de valores menores que 15? Qual a frequência de valores maiores que 10 e menores que 15?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def mostrar_totais(tabela, colunas):
    """Mostra Σ no rodapé sem acrescentar linhas ao DataFrame de cálculo."""
    rodape = {col: (tabela[col].sum() if col in colunas else '') for col in tabela.columns}
    rodape[tabela.columns[0]] = 'Σ'
    display(pd.concat([tabela, pd.DataFrame([rodape])], ignore_index=True))


### Dados brutos e rol

A variável é discreta; ainda assim, o enunciado pede **também** a representação em intervalos de cinco inteiros.


In [ ]:
dados = [8,11,8,12,14,13,11,14,14,15,6,10,14,19,6,12,7,5,8,8,10,16,10,12,12,8,11,6,7,12,7,10,14,5,12,7,9,12,11,9,14,8,14,8,12,10,12,22,7,15]
serie = pd.Series(dados, name='Erros')
n = len(serie)
print('n =', n)
print('Rol:', sorted(dados))


### Distribuição individual

$ f_i $ conta cada valor; $fr_i=f_i/n$; $F_i$ e $Fr_i$ acumulam as frequências. As colunas auxiliares deixam visíveis as parcelas das somas da média e dos momentos.


In [ ]:
tabela = (serie.value_counts().sort_index().rename_axis('xi').reset_index(name='fi'))
tabela['fri'] = tabela.fi / n
tabela['Fac'] = tabela.fi.cumsum()
tabela['Fra'] = tabela.fri.cumsum()
media = (tabela.xi*tabela.fi).sum()/n
tabela['xi·fi'] = tabela.xi*tabela.fi
tabela['fi·(xi−média)²'] = tabela.fi*(tabela.xi-media)**2
tabela['fi·(xi−média)³'] = tabela.fi*(tabela.xi-media)**3
mostrar_totais(tabela, ['fi','fri','xi·fi','fi·(xi−média)²','fi·(xi−média)³'])


### Medidas calculadas

Usaremos a variância **populacional** (divisor $n$), coerente com a descrição destes dados. Para dados agrupados, os pontos médios aproximam as observações:

$$\bar x=\frac{\sum f_ix_i}{n},\qquad
\sigma^2=\frac{\sum f_i(x_i-\bar x)^2}{n},\qquad
\sigma=\sqrt{\sigma^2},\qquad
g_1=\frac{\sum f_i(x_i-\bar x)^3}{n\sigma^3}.$$

O coeficiente de variação é $CV=\sigma/\bar x$ (em porcentagem, multiplicar por 100).


In [ ]:
variancia = tabela['fi·(xi−média)²'].sum()/n
desvio = np.sqrt(variancia)
assimetria = tabela['fi·(xi−média)³'].sum()/(n*desvio**3)
print(f'Média = {media:.4f}; variância = {variancia:.4f}; desvio = {desvio:.4f}')
print(f'Assimetria = {assimetria:.4f}; CV = {desvio/media:.2%}')


### Distribuição por classes

Limites $[5,10)$, $[10,15)$, $[15,20)$ e $[20,25)$; para contagens inteiras, equivalem a 5–9, 10–14, 15–19 e 20–24. As frequências de classe vêm diretamente dos dados.


In [ ]:
faixas = [5,10,15,20,25]
classe = pd.cut(serie, bins=faixas, right=False)
tabela_classes = (classe.value_counts(sort=False).rename_axis('Classe').reset_index(name='fi'))
tabela_classes['fri'] = tabela_classes.fi/n
tabela_classes['Fac'] = tabela_classes.fi.cumsum()
tabela_classes['Fra'] = tabela_classes.fri.cumsum()
mostrar_totais(tabela_classes, ['fi','fri'])


### Perguntas e gráfico

As desigualdades são estritas: 10 e 15 não entram no intervalo $10<x<15$.


In [ ]:
for rotulo, mascara in [('x > 10',serie>10),('x < 15',serie<15),('10 < x < 15',(serie>10)&(serie<15))]:
    print(f'{rotulo}: {mascara.sum()} dias ({mascara.mean():.0%})')
print(f'Mediana = {serie.median()}; modas = {serie.mode().tolist()}')
print(f'Q1 = {serie.quantile(.25)}; Q3 = {serie.quantile(.75)} (interpolação linear do pandas)')
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(11,4))
ax1.bar(tabela.xi,tabela.fri); ax1.set(xlabel='Erros',ylabel='Frequência relativa',title='Frequências individuais')
ax2.plot(tabela.xi,tabela.Fra,marker='o'); ax2.set(xlabel='Erros',ylabel='Frequência relativa acumulada',title='Acumulada',ylim=(0,1.05))
for ax in (ax1,ax2): ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(7,2.5)); ax.boxplot(serie,vert=False,showmeans=True); ax.set(xlabel='Erros',title='Boxplot'); ax.grid(True,alpha=.3); plt.show()


**Leitura:** a tabela por classes resume as contagens; média, desvio e assimetria acima usam os valores individuais, sem aproximação pelos pontos médios.
